# E5 — Demo: Pipeline completo E2→E3→E4

Notebook de demostración end-to-end del pipeline de reparación 3D:

```
[Vóxeles E2]  ──►  convertir_voxels_a_nube  ──►  [Nube rota (2048,3)]
                                                         │
                                                         ▼
                                              PoinTr  (E3 shape completion)
                                                         │
                                                         ▼
                                             [Nube completa (2048+N,3)]
                                                         │
                                                         ▼
                                        Poisson + manifold3d  (E4)
                                                         │
                                                         ▼
                                                 [STL imprimible]
```

**Modo de uso:**
- **Con vóxeles reales de E2**: sube el fichero `.npy` de salida de Pix2Vox++ → la Celda 3 lo convierte.
- **Sin E2 todavía** (demo): la Celda 3 usa una nube de puntos sintética del dataset de test.

Al final, la Celda 9 lanza una **aplicación Gradio** accesible desde el navegador.

---
## Sección 1 — Instalación y clonado

In [1]:
import subprocess, os
from getpass import getpass

for pkg in ['trimesh', 'manifold3d', 'pymeshlab', 'plotly', 'gradio',
            'scipy', 'numpy', 'easydict', 'timm', 'opencv-python-headless']:
    r = subprocess.run(['pip', 'install', pkg, '-q'], capture_output=True, text=True)
    print(f'[{"OK" if r.returncode==0 else "WARN"}] {pkg}')

# Clonar repos
if not os.path.exists('/content/PoinTr'):
    subprocess.run(['git','clone','https://github.com/yuxumin/PoinTr',
                    '/content/PoinTr','--depth=1','-q'], capture_output=True)
    print('PoinTr clonado.')

REPO = '/content/TFM'
if not os.path.exists(REPO):
    token = getpass('Token GitHub (ghp_...): ')
    subprocess.run(['git','clone',f'https://{token}@github.com/herredoble/TFM-reconstruccion-3D',
                    REPO,'-q'], capture_output=True)
    del token; print('Repo clonado.')
else:
    subprocess.run(['git','-C',REPO,'pull','-q'], capture_output=True)
    print('[OK] Repo actualizado.')

os.chdir(REPO)
subprocess.run(['git','checkout','raquel/e3','-q'], capture_output=True)
print('[OK] listo')

[OK] trimesh
[OK] manifold3d
[OK] pymeshlab
[OK] plotly
[OK] gradio
[OK] scipy
[OK] numpy
[OK] easydict
[OK] timm
[OK] opencv-python-headless
PoinTr clonado.
Token GitHub (ghp_...): ··········
Repo clonado.
[OK] listo


---
## Sección 2 — Cargar modelo PoinTr (E3)

In [2]:
import sys, types, glob as _glob, importlib
import torch, torch.nn as nn, numpy as np
from pathlib import Path
from easydict import EasyDict
from google.colab import drive

if not (Path('/content/drive').exists() and list(Path('/content/drive').iterdir())):
    drive.mount('/content/drive')
    print('Drive montado.')

DRIVE      = '/content/drive/MyDrive'
VERSION_E3 = 'v6_obj_sn'
BASE_E3    = f'{DRIVE}/Datos_E2_E3/E3/Raquel'
DRIVE_E5   = f'{DRIVE}/E5'   # donde está convertir_voxels_a_nube.py en Drive

# ── Sys path ─────────────────────────────────────────────────
for p in ['/content/TFM', '/content/PoinTr']:
    if p in sys.path: sys.path.remove(p)
for p in [DRIVE_E5, '/content/TFM', '/content/PoinTr']:
    if p not in sys.path:
        sys.path.insert(0, p)
importlib.invalidate_caches()
os.chdir('/content/TFM')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE_STR = str(device)
print(f'Dispositivo: {device}')
print(f'Checkpoint : {BASE_E3}/modelos/{VERSION_E3}/best.pt')
print(f'Script conv: {DRIVE_E5}/convertir_voxels_a_nube.py')

# ── Mocks CUDA (igual que en entrenamiento) ───────────────────
_pfx = ('models','utils.registry','utils.config','utils.logger','utils.misc','extensions')
_del = [k for k,v in sys.modules.items()
        if any(k==p or k.startswith(p+'.') for p in _pfx)
        or (hasattr(v,'__file__') and v.__file__ and '/content/PoinTr' in str(v.__file__))]
for k in _del: del sys.modules[k]
importlib.invalidate_caches()

for fp in _glob.glob('/content/PoinTr/models/*.py')+_glob.glob('/content/PoinTr/models/**/*.py'):
    try:
        src=open(fp,encoding='utf-8').read(); new=src.replace('.cuda()',f'.to("{DEVICE_STR}")')
        if new!=src: open(fp,'w',encoding='utf-8').write(new)
    except: pass

def _force(n,a):
    m=types.ModuleType(n)
    for k,v in a.items(): setattr(m,k,v)
    sys.modules[n]=m

def _cr(a,b): d=torch.cdist(a,b,p=2); return d.min(2).values,d.min(1).values
class _CL1(nn.Module):
    def forward(self,a,b): d1,d2=_cr(a.contiguous(),b.contiguous()); return (d1.mean()+d2.mean())/2
_ch={'ChamferDistanceL1':_CL1,'ChamferDistanceL2':_CL1,'ChamferDistanceL1_PM':_CL1,'chamfer_3DDist':_cr}
for n in ['chamfer','chamfer_dist','extensions.chamfer_dist','chamfer3D','chamfer3D.dist_chamfer_3D']:
    _force(n,_ch)

if 'pointnet2_ops' not in sys.modules:
    def _fps(xyz,np_):
        B,N,_=xyz.shape; dev=xyz.device
        idx=torch.zeros(B,np_,dtype=torch.int32,device=dev); dist=torch.full((B,N),1e10,device=dev)
        far=torch.randint(0,N,(B,),dtype=torch.long,device=dev); bi=torch.arange(B,dtype=torch.long,device=dev)
        for i in range(np_):
            idx[:,i]=far.int(); c=xyz[bi,far].unsqueeze(1)
            dist=torch.min(dist,((xyz-c)**2).sum(-1)); far=dist.max(-1)[1]
        return idx
    def _go(f,idx): B,C,N=f.shape; M=idx.shape[1]; return f.gather(2,idx.long().unsqueeze(1).expand(B,C,M)).contiguous()
    def _bq(r,ns,xyz,nxyz):
        d=torch.cdist(nxyz.float(),xyz.float()); s=d.argsort(-1)[:,:,:ns]
        return torch.where(d.gather(2,s)>r,s[:,:,:1].expand_as(s),s).int()
    def _grp(f,idx):
        B,C,N=f.shape; S,K=idx.shape[1],idx.shape[2]
        return f.gather(2,idx.long().view(B,1,S*K).expand(B,C,S*K)).view(B,C,S,K).contiguous()
    def _3nn(u,k): d=torch.cdist(u.float(),k.float()); d2,i=d.topk(3,-1,largest=False); return d2.float(),i.int()
    def _3i(f,idx,w):
        B,C,M=f.shape; N=idx.shape[1]
        return (f.gather(2,idx.long().view(B,1,N*3).expand(B,C,N*3)).view(B,C,N,3)*w.unsqueeze(1)).sum(-1).contiguous()
    _pu=types.ModuleType('pointnet2_ops.pointnet2_utils')
    for k,v in {'furthest_point_sample':_fps,'gather_operation':_go,'ball_query':_bq,
                'grouping_operation':_grp,'three_nn':_3nn,'three_interpolate':_3i}.items(): setattr(_pu,k,v)
    _pm=types.ModuleType('pointnet2_ops'); _pm.pointnet2_utils=_pu
    sys.modules['pointnet2_ops']=_pm; sys.modules['pointnet2_ops.pointnet2_utils']=_pu

class _KNN(nn.Module):
    def __init__(self,k,transpose_mode=False): super().__init__(); self.k=k; self.tm=transpose_mode
    def forward(self,ref,query):
        if self.tm: d=torch.cdist(query.float(),ref.float()); dk,ik=d.topk(self.k,-1,largest=False); return dk,ik
        r=ref.transpose(1,2).contiguous(); q=query.transpose(1,2).contiguous()
        d=torch.cdist(q.float(),r.float()); dk,ik=d.topk(self.k,-1,largest=False)
        return dk.transpose(1,2),ik.transpose(1,2)
_force('knn_cuda',{'KNN':_KNN})

def _dc(n): return type(n,(nn.Module,),{'__init__':lambda s,*a,**k:super(type(s),s).__init__(),'forward':lambda s,x,*a,**k:x})
class _Emd(nn.Module):
    def forward(self,a,b): return torch.zeros(a.shape[0],device=a.device),torch.zeros(a.shape[0],dtype=torch.int32,device=a.device)
for base,attrs in [('gridding',{'Gridding':_dc('G'),'GriddingReverse':_dc('GR')}),
                   ('gridding_loss',{'GriddingLoss':_dc('GL')}),
                   ('cubic_feature_sampling',{'CubicFeatureSampling':_dc('CFS')}),
                   ('emd',{'emd_module':_Emd,'EarthMoverDistance':_Emd})]:
    for pfx in ['','extensions.']:
        if pfx+base not in sys.modules: _force(pfx+base,attrs)

# ── Cargar checkpoint ─────────────────────────────────────────
ckpt_local = f'E3/checkpoints_pointr_{VERSION_E3}/best.pt'
ckpt_drive  = f'{BASE_E3}/modelos/{VERSION_E3}/best.pt'
ckpt_path   = ckpt_local if Path(ckpt_local).exists() else ckpt_drive
print(f'Cargando checkpoint desde: {ckpt_path}')
ck = torch.load(ckpt_path, map_location=device, weights_only=False)
print(f'PoinTr {VERSION_E3} — best epoch {ck["epoch"]}')

try:
    from models.build import build_model_from_cfg
    model = build_model_from_cfg(EasyDict(ck['model_cfg']))
except:
    from models.PoinTr import PoinTr
    model = PoinTr(EasyDict(ck['model_cfg']))
model.load_state_dict(ck['model_state_dict'])
model = model.to(device).eval()
print(f'Modelo listo. Params: {sum(p.numel() for p in model.parameters()):,}')

# ── Importar script de conversión E2→E3 ──────────────────────
# Busca primero en Drive/E5, luego en el repo clonado
try:
    from convertir_voxels_a_nube import voxels_a_nube, diagnosticar_voxels
    print('[OK] Script de conversión cargado desde Drive/E5')
except ImportError:
    from Scripts.convertir_voxels_a_nube import voxels_a_nube, diagnosticar_voxels
    print('[OK] Script de conversión cargado desde el repo')

Mounted at /content/drive
Drive montado.
Dispositivo: cuda
Checkpoint : /content/drive/MyDrive/Datos_E2_E3/E3/Raquel/modelos/v6_obj_sn/best.pt
Script conv: /content/drive/MyDrive/E5/convertir_voxels_a_nube.py
Cargando checkpoint desde: /content/drive/MyDrive/Datos_E2_E3/E3/Raquel/modelos/v6_obj_sn/best.pt
PoinTr v6_obj_sn — best epoch 486


/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
2026-09-12 10:15:01,479 - MODEL - INFO -  Transformer with knn_layer 1


Modelo listo. Params: 42,202,854
[OK] Script de conversión cargado desde Drive/E5


---
## Sección 2b — Cargar modelo Pix2Vox++ (E2)

In [3]:
import cv2 as _cv2
import sys as _sys

# Añadir E2 al path para importar modelo_pix2vox
_e2_path = '/content/TFM/E2'
if _e2_path not in _sys.path:
    _sys.path.insert(0, _e2_path)

from modelo_pix2vox import Pix2VoxPlusPlusA, cargar_checkpoint_finetuning

CHECKPOINT_E2 = 'exp13_7categorias_5v_finetuning_mejor.pth'
RUTA_CKPT_E2_DRIVE = f'{DRIVE}/Datos_E2_E3/E2/Pix2Vox++/checkpoints/{CHECKPOINT_E2}'
RUTA_CKPT_E2_LOCAL = f'/content/TFM/E2/checkpoints/{CHECKPOINT_E2}'

ruta_ckpt_e2 = RUTA_CKPT_E2_LOCAL if Path(RUTA_CKPT_E2_LOCAL).exists() else RUTA_CKPT_E2_DRIVE
print(f'Cargando Pix2Vox++ desde: {ruta_ckpt_e2}')

modelo_e2 = Pix2VoxPlusPlusA(usar_refiner=True, usar_merger=True, usar_pesos_imagenet=False)
meta_e2 = cargar_checkpoint_finetuning(modelo_e2, ruta_ckpt_e2, dispositivo_carga='cpu')
modelo_e2 = modelo_e2.to(device).eval()
UMBRAL_E2 = meta_e2.get('mejor_umbral') or 0.3
print(f'Pix2Vox++ listo | época={meta_e2.get("epoca")} | umbral={UMBRAL_E2}')

# Preprocesado de imágenes
_FONDO_E2 = 240 / 255.0

def _cargar_img_e2(ruta):
    img = _cv2.imread(str(ruta), _cv2.IMREAD_UNCHANGED)
    if img is None:
        raise ValueError(f'No se pudo cargar: {ruta}')
    if img.ndim == 2:
        img = _cv2.cvtColor(img, _cv2.COLOR_GRAY2BGR)
    if img.shape[2] == 4:
        alpha = img[:, :, 3:4].astype(np.float32) / 255.0
        bgr   = img[:, :, :3].astype(np.float32) / 255.0
        bgr   = bgr * alpha + _FONDO_E2 * (1 - alpha)
    else:
        bgr = img[:, :, :3].astype(np.float32) / 255.0
    bgr = _cv2.resize(bgr, (224, 224), interpolation=_cv2.INTER_LINEAR)
    bgr = (bgr - 0.5) / 0.5
    return torch.tensor(bgr.transpose(2, 0, 1), dtype=torch.float32)

@torch.no_grad()
def inferir_voxel_e2(rutas_imagenes):
    tensors = [_cargar_img_e2(r) for r in rutas_imagenes]
    imgs = torch.stack(tensors).unsqueeze(0).to(device)  # (1,5,3,224,224)
    return modelo_e2(imgs)['volumen_final'].squeeze().cpu().numpy()

print('Funciones E2 listas (inferir_voxel_e2).')

INFRAESTRUCTURA DE CHECKPOINTS DEFINIDA
Conversión segura de metadatos: OK
Guardado de checkpoints:        OK
Carga de checkpoints:           OK
Compatibilidad con checkpoint v1: OK
Cargando Pix2Vox++ desde: /content/drive/MyDrive/Datos_E2_E3/E2/Pix2Vox++/checkpoints/exp13_7categorias_5v_finetuning_mejor.pth
Pix2Vox++ listo | época=29 | umbral=0.3
Funciones E2 listas (inferir_voxel_e2).


---
## Sección 3 — Cargar nube rota (desde E2 o demo sintética)

In [4]:
# ══════════════════════════════════════════════════════════════
# OPCION A — Vóxeles de E2 (Pix2Vox++): subir fichero .npy
# ══════════════════════════════════════════════════════════════
USAR_VOXELES_E2 = False  # ← cambia a True si tienes salida de Pix2Vox++

if USAR_VOXELES_E2:
    from google.colab import files
    print('Sube el fichero .npy de vóxeles de E2 (shape 32×32×32):')
    subidos = files.upload()
    ruta_voxels = list(subidos.keys())[0]

    # voxels_a_nube y diagnosticar_voxels ya fueron importados en la Sección 2
    voxels = np.load(ruta_voxels)
    diag = diagnosticar_voxels(voxels)
    print(f'Grid: {diag["forma_grid"]}  ocupado: {diag["porcentaje_ocupado"]}%')
    for w in diag['advertencias']:
        if w: print(f'  ⚠️  {w}')

    nube_rota = voxels_a_nube(voxels, n_puntos=2048)
    NOMBRE_OBJETO = Path(ruta_voxels).stem
    print(f'Nube generada desde vóxeles: {nube_rota.shape}  radio_max={np.linalg.norm(nube_rota,axis=1).max():.3f}')

else:
    # ══════════════════════════════════════════════════════════
    # OPCION B — Demo: usar una nube del test set de E3
    # ══════════════════════════════════════════════════════════
    import E3.dataset as _ds; _ds.CENTRAR_EN_ROTO = False
    from E3.dataset import construir_pares
    import random

    BASE_GEN = f'{DRIVE}/Datos_E2_E3/General'
    carpetas = [f'{BASE_GEN}/shapenet_roturas', f'{BASE_GEN}/roturas_Objaverse_v2']
    carpetas_ok = [c for c in carpetas if Path(c).exists()]

    if not carpetas_ok:
        carpetas_ok = [c for c in ['Datos/shapenet/roturas', 'Datos/objaverse/roturas_v2']
                       if Path(c).exists()]

    todos = construir_pares(carpetas_ok)
    rng = random.Random(42); rng.shuffle(todos)
    n = len(todos); _te = todos[int(0.9*n):]

    INDICE_DEMO = 0   # ← cambia para ver otros objetos del test set
    ruta_roto, ruta_comp = _te[INDICE_DEMO]
    nube_rota = np.load(ruta_roto).astype(np.float32)
    nube_gt   = np.load(ruta_comp).astype(np.float32)
    NOMBRE_OBJETO = Path(ruta_roto).stem.replace('_roto','')
    print(f'Objeto de demo: {NOMBRE_OBJETO}')
    print(f'Nube rota: {nube_rota.shape}  radio_max={np.linalg.norm(nube_rota,axis=1).max():.3f}')
    print(f'(GT disponible para comparación — no se usa como entrada al modelo)')

Objeto de demo: objaverse_fa9c188ebe9c49f994f6c5d6bac70ec6_r2
Nube rota: (2048, 3)  radio_max=1.000
(GT disponible para comparación — no se usa como entrada al modelo)


---
## Sección 4 — E3: PoinTr shape completion

In [5]:
def inferir_pointr(nube_np: np.ndarray) -> np.ndarray:
    """Nube rota (2048,3) → nube completa predicha (N,3)."""
    with torch.no_grad():
        inp = torch.tensor(nube_np, dtype=torch.float32).unsqueeze(0).to(device)
        out = model(inp)
        fine = out[-1] if isinstance(out, (list, tuple)) else out
        return fine.squeeze(0).cpu().numpy()

def filtrar_outliers(pts, k=20, std_ratio=1.5):
    from scipy.spatial import cKDTree
    tree = cKDTree(pts)
    dists, _ = tree.query(pts, k=k+1)
    mean_d = dists[:, 1:].mean(axis=1)
    umbral = mean_d.mean() + std_ratio * mean_d.std()
    mask = mean_d < umbral
    return pts[mask], int((~mask).sum())

def chamfer_l1(p, g):
    pt = torch.tensor(p).unsqueeze(0); gt = torch.tensor(g).unsqueeze(0)
    d = torch.cdist(pt, gt, p=2)
    return ((d.min(2).values.mean() + d.min(1).values.mean()) / 2).item()

# Inferencia E3
print('Ejecutando PoinTr...')
pred_raw = inferir_pointr(nube_rota)
pred, n_outliers = filtrar_outliers(pred_raw)
print(f'PoinTr: {pred_raw.shape[0]} pts → {pred.shape[0]} pts (eliminados {n_outliers} outliers)')

# Métricas (solo si tenemos GT)
if 'nube_gt' in dir() or 'nube_gt' in locals():
    cd_val = chamfer_l1(pred, nube_gt)
    print(f'CD-L1 vs GT: {cd_val:.4f}')

# Visualización de la nube completa predicha
import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Scatter3d(x=pred[:,0], y=pred[:,1], z=pred[:,2], mode='markers',
    name='PoinTr (completo)', marker=dict(size=2, color='#66BB6A', opacity=0.85)))
fig.add_trace(go.Scatter3d(x=nube_rota[:,0], y=nube_rota[:,1], z=nube_rota[:,2], mode='markers',
    name='Roto (entrada)', marker=dict(size=3, color='#EF5350', opacity=0.95)))
fig.update_layout(
    scene=dict(bgcolor='#111',
               xaxis=dict(showticklabels=False, backgroundcolor='#111', gridcolor='#333'),
               yaxis=dict(showticklabels=False, backgroundcolor='#111', gridcolor='#333'),
               zaxis=dict(showticklabels=False, backgroundcolor='#111', gridcolor='#333'),
               aspectmode='cube'),
    title=dict(text=f'<b>E3: Shape completion</b> — {NOMBRE_OBJETO}<br><sup>Rojo=roto | Verde=reconstruido por PoinTr</sup>',
               font=dict(color='white'), x=0.5),
    paper_bgcolor='#111', legend=dict(font=dict(color='white')),
    height=560, width=680, margin=dict(l=0,r=0,t=60,b=0))
fig.show()
print('E3 completado.')

Ejecutando PoinTr...
PoinTr: 4096 pts → 3958 pts (eliminados 138 outliers)
CD-L1 vs GT: 0.0386


E3 completado.


---
## Sección 5 — E4: Generación de STL

In [6]:
import pymeshlab, trimesh
from scipy.spatial import cKDTree

TAMANO_MM = 100.0

def suavizar_nube(pts, k=15, iters=3):
    tree = cKDTree(pts)
    _, idx = tree.query(pts, k=k+1)
    result = pts.copy()
    for _ in range(iters):
        result = result[idx[:, 1:]].mean(axis=1)
    return result.astype(np.float32)

def poisson_stl(pts, depth=7):
    ms = pymeshlab.MeshSet()
    ms.add_mesh(pymeshlab.Mesh(vertex_matrix=pts.astype(np.float64)))
    ms.compute_normal_for_point_clouds(k=20, smoothiter=2)
    ms.generate_surface_reconstruction_screened_poisson(depth=depth, scale=1.1)
    ms.meshing_remove_connected_component_by_face_number(mincomponentsize=200)
    ms.apply_coord_laplacian_smoothing(stepsmoothnum=2)
    m = ms.current_mesh()
    return trimesh.Trimesh(vertices=m.vertex_matrix(), faces=m.face_matrix(), process=False)

def reparar(mesh):
    comps = mesh.split(only_watertight=False)
    if len(comps) > 1:
        mesh = max(comps, key=lambda c: len(c.faces))
    try:
        import manifold3d
        m = manifold3d.Manifold(manifold3d.Mesh(
            vert_properties=np.array(mesh.vertices, dtype=np.float32),
            tri_verts=np.array(mesh.faces, dtype=np.uint32)))
        out = m.to_mesh()
        res = trimesh.Trimesh(vertices=np.array(out.vert_properties),
                              faces=np.array(out.tri_verts), process=False)
        if len(res.vertices) > 0:
            mesh = res
    except: pass
    trimesh.repair.fill_holes(mesh)
    trimesh.repair.fix_normals(mesh)
    mesh.process(validate=False)
    return mesh

# Pipeline E4
print('Ejecutando E4...')
pred_suav = suavizar_nube(pred, k=15, iters=3)
print(f'  Suavizado Laplaciano: {pred.shape[0]} pts')

mesh_raw = poisson_stl(pred_suav, depth=7)
print(f'  Poisson depth=7: {len(mesh_raw.faces):,} caras')

mesh_raw.apply_translation(-mesh_raw.centroid)
lado = mesh_raw.bounding_box.extents.max()
if lado > 0: mesh_raw.apply_scale(TAMANO_MM / lado)

mesh_final = reparar(mesh_raw)
wt = bool(mesh_final.is_watertight)
eu = int(mesh_final.euler_number)
print(f'  Reparación: {len(mesh_final.faces):,} caras | watertight={wt} | euler={eu}')
print()
print('═'*50)
print(f'RESULTADO: {"✅ WATERTIGHT" if wt else "⚠️  No watertight"}')
print(f'  Caras   : {len(mesh_final.faces):,}')
print(f'  Euler   : {eu}  (0=esfera, ±2=taza con/sin asa)')
print(f'  Tamaño  : {TAMANO_MM:.0f} mm lado mayor')
print('═'*50)

# Guardar STL
Path('E5').mkdir(exist_ok=True)
ruta_stl = f'E5/{NOMBRE_OBJETO}_demo.stl'
mesh_final.export(ruta_stl)
print(f'STL guardado: {ruta_stl}')

# Visualización STL
verts = np.array(mesh_final.vertices)
faces = np.array(mesh_final.faces)
if len(faces) > 0:
    z = verts[:,2]
    intensidad = (z - z.min()) / (np.ptp(z) + 1e-8)
    fig2 = go.Figure(go.Mesh3d(
        x=verts[:,0], y=verts[:,1], z=verts[:,2],
        i=faces[:,0], j=faces[:,1], k=faces[:,2],
        intensity=intensidad,
        colorscale=[[0,'#0D47A1'],[0.5,'#29B6F6'],[1,'#E1F5FE']],
        showscale=False,
        lighting=dict(ambient=0.3,diffuse=0.85,roughness=0.3,specular=0.6),
        lightposition=dict(x=200,y=300,z=400)))
    fig2.update_layout(
        scene=dict(bgcolor='#111',
                   xaxis=dict(showticklabels=False,backgroundcolor='#111',gridcolor='#333'),
                   yaxis=dict(showticklabels=False,backgroundcolor='#111',gridcolor='#333'),
                   zaxis=dict(showticklabels=False,backgroundcolor='#111',gridcolor='#333'),
                   aspectmode='data'),
        title=dict(text=f'<b>E4: STL final</b> — {NOMBRE_OBJETO}<br>'
                        f'<sup>{"WATERTIGHT" if wt else "no watertight"} | euler={eu} | {len(faces):,} caras</sup>',
                   font=dict(color='white'), x=0.5),
        paper_bgcolor='#111', height=560, width=680,
        margin=dict(l=0,r=0,t=60,b=0))
    fig2.show()

Ejecutando E4...
  Suavizado Laplaciano: 3958 pts
  Poisson depth=7: 29,017 caras
  Reparación: 29,017 caras | watertight=False | euler=-12

══════════════════════════════════════════════════
RESULTADO: ⚠️  No watertight
  Caras   : 29,017
  Euler   : -12  (0=esfera, ±2=taza con/sin asa)
  Tamaño  : 100 mm lado mayor
══════════════════════════════════════════════════
STL guardado: E5/objaverse_fa9c188ebe9c49f994f6c5d6bac70ec6_r2_demo.stl


---
## Sección 6 — Descargar STL

In [7]:
from google.colab import files
if Path(ruta_stl).exists():
    print(f'Descargando {ruta_stl}...')
    files.download(ruta_stl)
else:
    print('STL no encontrado — ejecuta la Sección 5 primero.')

Descargando E5/objaverse_fa9c188ebe9c49f994f6c5d6bac70ec6_r2_demo.stl...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## Seccion 7 — App REBUILD3D

Lanza la interfaz **REBUILD3D** en el navegador.

**Tab 1 — Pipeline completo:** sube 5 fotos del objeto roto → E2 (Pix2Vox++) → E3 (PoinTr) → E4 (STL imprimible)

**Tab 2 — Desde .npy:** voxeles (32×32×32) o nube de puntos (N×3) → E3 → E4

Cada etapa muestra una **visualizacion 3D interactiva** en su propia pestana.

> Si Colab muestra un enlace `*.gradio.live`, abrelo en cualquier navegador.

In [8]:
import gradio as gr
import tempfile
import plotly.graph_objects as go

# ── Helpers de visualización 3D ───────────────────────────────────────────────
_BG = '#0F1117'
_AXIS_STYLE = dict(
    showticklabels=False, backgroundcolor=_BG,
    gridcolor='#2D3250', showspikes=False, zeroline=False
)
_SCENE_BASE = dict(
    bgcolor=_BG,
    xaxis=_AXIS_STYLE, yaxis=_AXIS_STYLE, zaxis=_AXIS_STYLE,
    aspectmode='cube',
    camera=dict(eye=dict(x=1.4, y=1.4, z=0.8))
)
_LAYOUT_BASE = dict(
    paper_bgcolor=_BG, plot_bgcolor=_BG,
    legend=dict(font=dict(color='#CFD8DC', size=12), bgcolor='rgba(0,0,0,0.4)',
                bordercolor='#37474F', borderwidth=1),
    margin=dict(l=0, r=0, t=48, b=0), height=430,
    font=dict(color='#CFD8DC')
)

def _placeholder_fig(mensaje='Resultado aparecerá aquí tras ejecutar el pipeline'):
    return go.Figure(layout=go.Layout(
        paper_bgcolor=_BG, plot_bgcolor=_BG, height=430,
        annotations=[dict(text=mensaje, showarrow=False,
                          font=dict(color='#455A64', size=14),
                          xref='paper', yref='paper', x=0.5, y=0.5)]
    ))

def _fig_voxeles(voxel_prob, umbral=0.3, titulo=''):
    voxel_bin = voxel_prob >= umbral
    zz, yy, xx = np.where(voxel_bin)
    if len(xx) == 0:
        return _placeholder_fig('Sin voxeles activos (umbral demasiado alto)')
    coords = np.stack([xx, yy, zz], axis=1).astype(float)
    coords = (coords / (voxel_prob.shape[0] - 1)) * 2 - 1  # [0,N-1] -> [-1,1]
    probs = voxel_prob[voxel_bin]
    fig = go.Figure(go.Scatter3d(
        x=coords[:,0], y=coords[:,1], z=coords[:,2],
        mode='markers',
        name=f'Voxeles activos ({voxel_bin.sum():,} / {voxel_prob.size:,})',
        marker=dict(
            size=3.5,
            color=probs,
            colorscale=[[0,'#1565C0'],[0.5,'#00BCD4'],[1,'#80DEEA']],
            opacity=0.8,
            showscale=True,
            colorbar=dict(title=dict(text='P(ocupado)', font=dict(color='#CFD8DC')),
                          tickfont=dict(color='#CFD8DC'), thickness=12, len=0.6)
        )
    ))
    fig.update_layout(
        scene=_SCENE_BASE,
        title=dict(text=titulo, font=dict(color='#CFD8DC', size=13), x=0.5),
        **_LAYOUT_BASE
    )
    return fig

def _fig_nube(roto, completo=None, titulo=''):
    fig = go.Figure()
    fig.add_trace(go.Scatter3d(
        x=roto[:,0], y=roto[:,1], z=roto[:,2],
        mode='markers', name='Entrada (roto)',
        marker=dict(size=2.5, color='#EF5350', opacity=0.9)
    ))
    if completo is not None:
        fig.add_trace(go.Scatter3d(
            x=completo[:,0], y=completo[:,1], z=completo[:,2],
            mode='markers', name='Reconstruido',
            marker=dict(size=2, color='#00BCD4', opacity=0.75)
        ))
    fig.update_layout(
        scene=_SCENE_BASE,
        title=dict(text=titulo, font=dict(color='#CFD8DC', size=13), x=0.5),
        **_LAYOUT_BASE
    )
    return fig

def _fig_mesh(mesh, titulo=''):
    verts = np.array(mesh.vertices)
    faces = np.array(mesh.faces)
    if len(faces) == 0:
        return _placeholder_fig('Sin malla generada')
    z = verts[:,2]
    intens = (z - z.min()) / (np.ptp(z) + 1e-8)
    fig = go.Figure(go.Mesh3d(
        x=verts[:,0], y=verts[:,1], z=verts[:,2],
        i=faces[:,0], j=faces[:,1], k=faces[:,2],
        intensity=intens,
        colorscale=[[0,'#1565C0'], [0.45,'#00BCD4'], [1,'#E0F7FA']],
        showscale=False,
        lighting=dict(ambient=0.3, diffuse=0.9, roughness=0.3, specular=0.7),
        lightposition=dict(x=200, y=300, z=400)
    ))
    fig.update_layout(
        scene={**_SCENE_BASE, 'aspectmode': 'data'},
        title=dict(text=titulo, font=dict(color='#CFD8DC', size=13), x=0.5),
        **_LAYOUT_BASE
    )
    return fig

# ── Pipeline interno ──────────────────────────────────────────────────────────
def _normalizar_nube(nube):
    if len(nube) > 2048:
        idx = np.random.choice(len(nube), 2048, replace=False)
        nube = nube[idx]
    nube = nube - nube.mean(0)
    r = np.linalg.norm(nube, axis=1).max()
    if r > 1e-8:
        nube = nube / r
    return nube.astype(np.float32)

def _run_e3_e4(nube_rota):
    pred_raw = inferir_pointr(nube_rota)
    pred, n_out = filtrar_outliers(pred_raw)
    log_e3 = f'E3 PoinTr: {pred.shape[0]} pts (eliminados {n_out} outliers)'
    fig_e3 = _fig_nube(nube_rota, pred, 'E3 — Shape completion  |  rojo=roto · cyan=reconstruido')

    pred_suav = suavizar_nube(pred)
    mesh_raw = poisson_stl(pred_suav, depth=7)
    mesh_raw.apply_translation(-mesh_raw.centroid)
    lado = mesh_raw.bounding_box.extents.max()
    if lado > 0:
        mesh_raw.apply_scale(100.0 / lado)
    mesh_final = reparar(mesh_raw)
    wt = bool(mesh_final.is_watertight)
    eu = int(mesh_final.euler_number)
    estado = 'WATERTIGHT' if wt else 'no watertight'
    log_e4 = f'E4 STL: {len(mesh_final.faces):,} caras | {estado} | euler={eu} | 100 mm'
    titulo_e4 = f'E4 — Malla STL  |  {len(mesh_final.faces):,} caras  [{estado}]'
    fig_e4 = _fig_mesh(mesh_final, titulo_e4)

    tmp = tempfile.NamedTemporaryFile(suffix='.stl', delete=False, prefix='rebuild3d_')
    mesh_final.export(tmp.name)
    return tmp.name, log_e3 + '\n' + log_e4, fig_e3, fig_e4

# ── Funciones Gradio ──────────────────────────────────────────────────────────
_PH = _placeholder_fig()

def pipeline_imagenes(img1, img2, img3, img4, img5):
    imgs = [img1, img2, img3, img4, img5]
    if any(x is None for x in imgs):
        return None, 'Sube exactamente 5 imagenes (una por vista)', _PH, _PH, _PH, _PH
    try:
        rutas = [x if isinstance(x, str) else x.name for x in imgs]
        voxels = inferir_voxel_e2(rutas)
        umbral = UMBRAL_E2 if 'UMBRAL_E2' in dir() else 0.3
        ocupados = int((voxels >= umbral).sum())
        log = [f'E2 Pix2Vox++: grid {voxels.shape}  umbral={umbral}  voxeles activos={ocupados:,}']
        fig_vox = _fig_voxeles(voxels, umbral, f'E2 — Voxeles 32x32x32  (umbral={umbral})')
        nube_rota = voxels_a_nube(voxels, n_puntos=2048)
        log.append(f'Conversion voxel -> nube: {nube_rota.shape}')
        fig_nube = _fig_nube(nube_rota, titulo='E2 — Nube de puntos (2048 pts)')
        stl_path, log_rest, fig_e3, fig_e4 = _run_e3_e4(nube_rota)
        return stl_path, '\n'.join(log) + '\n' + log_rest, fig_vox, fig_nube, fig_e3, fig_e4
    except Exception as e:
        import traceback
        return None, f'Error:\n{e}\n\n{traceback.format_exc()}', _PH, _PH, _PH, _PH

def pipeline_npy(archivo_npy):
    if archivo_npy is None:
        return None, 'Sube un fichero .npy primero', _PH, _PH
    try:
        arr = np.load(archivo_npy.name).astype(np.float32)
        while arr.ndim > 3 and arr.shape[0] == 1:
            arr = arr[0]
        if arr.ndim == 3:
            diag = diagnosticar_voxels(arr)
            nube_rota = voxels_a_nube(arr, n_puntos=2048)
            log = [f'Voxeles {diag["forma_grid"]}  ({diag["porcentaje_ocupado"]}% ocupado)  ->  nube {nube_rota.shape}']
        elif arr.ndim == 2 and arr.shape[1] == 3:
            nube_rota = _normalizar_nube(arr)
            log = [f'Nube de puntos: {nube_rota.shape}  (normalizada)']
        else:
            return None, f'Formato no reconocido: shape={arr.shape}', _PH, _PH
        stl_path, log_rest, fig_e3, fig_e4 = _run_e3_e4(nube_rota)
        return stl_path, '\n'.join(log) + '\n' + log_rest, fig_e3, fig_e4
    except Exception as e:
        import traceback
        return None, f'Error:\n{e}\n\n{traceback.format_exc()}', _PH, _PH

# ── CSS y cabecera REBUILD3D ──────────────────────────────────────────────────
CSS = (
    '.rebuild3d-header {'
    '    background: linear-gradient(135deg, #0A1628 0%, #0D1F3C 60%, #091524 100%);'
    '    border: 1px solid #1E3A5F;'
    '    border-bottom: 2px solid #00BCD4;'
    '    padding: 28px 36px 20px;'
    '    border-radius: 14px;'
    '    margin-bottom: 4px;'
    '}'
    '.rebuild3d-logo {'
    '    font-size: 2.6em;'
    '    font-weight: 900;'
    '    letter-spacing: 0.15em;'
    '    background: linear-gradient(90deg, #00BCD4 0%, #4FC3F7 50%, #80DEEA 100%);'
    '    -webkit-background-clip: text;'
    '    -webkit-text-fill-color: transparent;'
    '    background-clip: text;'
    '    margin: 0 0 4px 0;'
    '    line-height: 1;'
    '}'
    '.rebuild3d-tagline {'
    '    color: #546E7A;'
    '    font-size: 0.88em;'
    '    letter-spacing: 0.06em;'
    '    margin: 0;'
    '}'
    '.rebuild3d-pipeline {'
    '    display: flex;'
    '    gap: 8px;'
    '    align-items: center;'
    '    margin-top: 10px;'
    '    flex-wrap: wrap;'
    '}'
    '.rb-stage {'
    '    background: rgba(0,188,212,0.1);'
    '    border: 1px solid rgba(0,188,212,0.3);'
    '    color: #80DEEA;'
    '    font-size: 0.78em;'
    '    font-weight: 600;'
    '    padding: 3px 12px;'
    '    border-radius: 20px;'
    '    letter-spacing: 0.05em;'
    '}'
    '.rb-arrow { color: #37474F; font-size: 1em; }'
    '.log-mono textarea {'
    '    font-family: "JetBrains Mono", "Fira Code", monospace !important;'
    '    font-size: 0.82em !important;'
    '    background: #0A0E18 !important;'
    '    color: #90A4AE !important;'
    '    border: 1px solid #1E2D3D !important;'
    '}'
    'footer { display: none !important; }'
)

HEADER_HTML = (
    '<div class="rebuild3d-header">'
    '<p class="rebuild3d-logo">REBUILD3D</p>'
    '<p class="rebuild3d-tagline">Reconstruccion y reparacion de objetos 3D rotos &nbsp;&middot;&nbsp; TFM UCM 2026</p>'
    '<div class="rebuild3d-pipeline">'
    '<span class="rb-stage">E2 &nbsp;Pix2Vox++</span>'
    '<span class="rb-arrow">&rarr;</span>'
    '<span class="rb-stage">E3 &nbsp;PoinTr</span>'
    '<span class="rb-arrow">&rarr;</span>'
    '<span class="rb-stage">E4 &nbsp;STL imprimible</span>'
    '</div>'
    '</div>'
)

# ── App Gradio ────────────────────────────────────────────────────────────────
with gr.Blocks(
    title='REBUILD3D',
    theme=gr.themes.Soft(
        primary_hue=gr.themes.colors.cyan,
        neutral_hue=gr.themes.colors.slate,
        font=[gr.themes.GoogleFont('Inter'), 'sans-serif']
    ),
    css=CSS
) as app:

    gr.HTML(HEADER_HTML)

    with gr.Tabs():

        # ── Tab 1: Pipeline completo desde imagenes ───────────────────────
        with gr.Tab('Pipeline completo (imagenes)'):
            gr.Markdown(
                '**Sube 5 fotos** del objeto roto desde distintos angulos. '
                'Fondo homogeneo (blanco o gris claro). '
                'Orden sugerido: frontal, lateral x2, trasera, superior.'
            )
            with gr.Row():
                imgs_in = [
                    gr.Image(type='filepath', label=f'Vista {i+1}', height=150)
                    for i in range(5)
                ]
            btn1 = gr.Button('Reconstruir desde imagenes', variant='primary', size='lg')

            with gr.Row():
                with gr.Column(scale=1, min_width=280):
                    stl_out1 = gr.File(label='STL generado', file_types=['.stl'])
                    log_out1 = gr.Textbox(
                        label='Log del pipeline', lines=12,
                        interactive=False, elem_classes=['log-mono']
                    )
                with gr.Column(scale=2):
                    with gr.Tabs():
                        with gr.Tab('E2 — Voxeles (32x32x32)'):
                            plot_vox = gr.Plot(value=_PH)
                        with gr.Tab('E2 — Nube de puntos'):
                            plot_e2 = gr.Plot(value=_PH)
                        with gr.Tab('E3 — Shape completion'):
                            plot_e3a = gr.Plot(value=_PH)
                        with gr.Tab('E4 — Malla STL'):
                            plot_e4a = gr.Plot(value=_PH)

            btn1.click(
                fn=pipeline_imagenes,
                inputs=imgs_in,
                outputs=[stl_out1, log_out1, plot_vox, plot_e2, plot_e3a, plot_e4a]
            )

        # ── Tab 2: Desde .npy ──────────────────────────────────────────────
        with gr.Tab('Desde .npy (sin E2)'):
            gr.Markdown(
                'Sube un `.npy` con **voxeles (32x32x32)** (salida de Pix2Vox++) '
                'o una **nube de puntos (Nx3)**.'
            )
            with gr.Row():
                with gr.Column(scale=1, min_width=280):
                    npy_in   = gr.File(label='Fichero .npy', file_types=['.npy'])
                    btn2     = gr.Button('Reconstruir desde .npy', variant='primary', size='lg')
                    stl_out2 = gr.File(label='STL generado', file_types=['.stl'])
                    log_out2 = gr.Textbox(
                        label='Log del pipeline', lines=12,
                        interactive=False, elem_classes=['log-mono']
                    )
                with gr.Column(scale=2):
                    with gr.Tabs():
                        with gr.Tab('E3 — Shape completion'):
                            plot_e3b = gr.Plot(value=_PH)
                        with gr.Tab('E4 — Malla STL'):
                            plot_e4b = gr.Plot(value=_PH)

            btn2.click(
                fn=pipeline_npy,
                inputs=[npy_in],
                outputs=[stl_out2, log_out2, plot_e3b, plot_e4b]
            )

app.launch(share=True, debug=False)
print('REBUILD3D lanzado -- abre el enlace de arriba en el navegador.')


/tmp/ipykernel_5965/938762736.py:255: UserWarning:

The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.



Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b94cf748f58e2856aa.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


REBUILD3D lanzado -- abre el enlace de arriba en el navegador.
